# Local defaults and plain mode

Current environment and world values are planning defaults. They do not allocate resources in this notebook process. `runtime.plain()` runs trusted local code inline with DRYML enforcement off; it is not worker isolation.

In [ ]:
import dryml
from dryml.core2 import Definition
from dryml.core2.object import Object
from dryml.dispatch import Dispatcher
from dryml.environments import CurrentEnvironmentSpec
from dryml.operations import make_function_call_spec
from dryml.worlds import LocalResourceInventory, WorldSpec


class PlainCounter(Object):
    def __init__(self, value):
        super().__init__()
        self.value = value

    def plus(self, value):
        return self.value + value



In [ ]:
requested_world = WorldSpec.from_data({'roles': {'main': {'replicas': 1, 'process': {}}}})
before_runtime = dryml.runtime.active_runtime()
previous_environment = dryml.environments.set_current(CurrentEnvironmentSpec())
previous_world = dryml.worlds.set_current(requested_world)
try:
    explanation = Dispatcher().explain(make_function_call_spec('operator:add', args=[1, 2]), inventory=LocalResourceInventory((0,)))
    assert explanation.launchable
    assert explanation.resolution.environment_selection.source == 'current'
    assert explanation.resolution.world_selection.source == 'current'
    assert dryml.runtime.active_runtime() is before_runtime

    with dryml.runtime.plain():
        counter = Definition(PlainCounter, 3).build()
        assert counter.plus(4) == 7
    assert dryml.runtime.active_runtime() is before_runtime

    try:
        with dryml.runtime.plain():
            raise RuntimeError('handled demonstration failure')
    except RuntimeError:
        pass
    assert dryml.runtime.active_runtime() is before_runtime
finally:
    if previous_environment is None:
        dryml.environments.reset_current()
    else:
        dryml.environments.set_current(previous_environment)
    if previous_world is None:
        dryml.worlds.reset_current()
    else:
        dryml.worlds.set_current(previous_world)
